# Rede neural com TensorFlow

## Classificação binária para diagnóstico de câncer de mama

Base de dados: Breast Cancer Wisconsin (Diagnostic)
https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import sklearn
import scikeras
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from scikeras.wrappers import KerasClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras import backend as k

In [2]:
pd.__version__, tf.__version__, sklearn.__version__, scikeras.__version__

('3.0.6', '2.18.1', '1.8.0', '0.13.0')

### Carregar os dados

**Carregue a base de dados, faça a divisão de treino e teste (para isso, utilize a função train_test_split do sklearn), como o tamanho da base de teste de 0.25.**

In [3]:
X = pd.read_csv('dados/dados_breast.csv')
y = pd.read_csv('dados/rotulos_breast.csv')

X_treinamento, X_teste, y_treinamento, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=0
)

In [4]:
X_treinamento.shape, y_treinamento.shape

((426, 30), (426, 1))

In [5]:
X_teste.shape, y_teste.shape

((143, 30), (143, 1))

### Estrutura da Rede Neural Artificial e Teste

In [6]:
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import cross_val_score
from tensorflow.keras.models import Sequential
import tensorflow as tf
from tensorflow.keras import backend as k

**Crie a RNA com as seguintes configurações:**

**Uma camada de entrada com 30 neurônios;**

**Uma camada oculta densa com 16 neurônios. Porque a camada oculta tem essa quantidade de neurônios?**

Foram usados 16 neurônios como um ponto intermediário entre as 30 entradas e a única saída. Esse valor não é uma regra e pode ser comparado com outras configurações.

**Adicione na camada oculta a função de ativação relu e inicialize os pesos com Random uniform initializer;**

**Adicione a camada de saída com a função de ativação sigmoid. Porque foi utilizada esta função para a saída da RNA?**

A sigmoid limita a saída ao intervalo entre 0 e 1. Neste problema binário, esse valor é tratado como a probabilidade da classe positiva.

**Porque utilizamos a classe chamada Sequential para a RNA?**

A classe `Sequential` é adequada aqui porque as camadas formam uma sequência simples, sem ramificações.

In [7]:
k.clear_session()
rede_neural = Sequential([
    tf.keras.layers.InputLayer(shape=(30,)),
    tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
    tf.keras.layers.Dense(units=1, activation='sigmoid')
])

**A partir da RNA gerada, explique o que são os valores apresentados na tabela da rede_neural.summary()**

- **Layer (type)**: Nome e tipo de cada camada da rede.
- **Output Shape**: Dimensão da saída de cada camada. `(None, 16)` significa que para cada amostra (None = qualquer quantidade de amostras) há 16 saídas.
- **Param #**: Número de parâmetros treináveis. Para a camada densa de 16 neurônios: (30 entradas × 16 neurônios) + 16 biases = 496 parâmetros. Para a camada de saída: (16 × 1) + 1 bias = 17 parâmetros. Total: 513 parâmetros.
- **Trainable params**: parâmetros que serão atualizados durante o treinamento.
- **Non-trainable params**: parâmetros fixos (ex.: em camadas congeladas).

In [8]:
rede_neural.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 513 (2.00 KB)

 Trainable params: 513 (2.00 KB)

 Non-trainable params: 0 (0.00 B)

**Adicione um otimizador Adam e especifique a classe loss binário - binary crossentropy e a métrica de acurácia binária.**

**Para que servem os otimizadores? Como o otimizador Adam funciona?**

Os otimizadores servem para atualizar os pesos da rede neural de forma a minimizar a função de perda (loss). O Adam (Adaptive Moment Estimation) combina duas técnicas:
- **Momentum**: acumula gradientes passados para suavizar a atualização (evita oscilações);
- **RMSProp**: adapta a taxa de aprendizado para cada parâmetro individualmente com base na magnitude dos gradientes recentes.
Na prática, isso permite ajustar o passo de cada parâmetro durante o treinamento.

**Depois de estruturados os parâmetros da RNA, utilize a função .fit para fazer o treinamento da rede. Com batch_size=10, quantos batches serão utilizados ao total?**

O conjunto de treinamento tem 426 amostras (75% de 569). Com batch_size=10: ⌈426 / 10⌉ = 43 batches por época × 100 épocas = 4.300 atualizações de pesos no total.

**Defina o número de épocas em que ocorre o treinamento igual a 100.**

In [9]:
rede_neural.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['binary_accuracy']
)

In [10]:
rede_neural.fit(X_treinamento, y_treinamento, batch_size=10, epochs=100, verbose=0)

**Crie uma variável chamada previsoes para realizar a previsão dos dados de teste (X_teste). O resultado da rede deve ser um valor entre 0 e 1. Porque isso acontece?**

A saída fica entre 0 e 1 por causa da sigmoid e é interpretada como a probabilidade da classe positiva.

In [11]:
previsoes = rede_neural.predict(X_teste, verbose=0)

**Converta o resultado que está em probabilidade para valores binários com um limiar (threshold) de 0.5.**

In [12]:
previsoes = previsoes > 0.5

In [13]:
previsoes

array([[False],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [False],
       [False],
       [ True],
       [False],
       [ True],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [ True],
       [ True],
       [False],
       [ True],
       [ True],
       [ True],
       [ True],
       [False],
       [ True],
       [False],
       [ True],
       [False],
       [ True],
       [False],
       [ True],
       [False],
       [ True],
       [False],
       [False],
       [ True],
       [False],
       [ True],
       [False],
       [False],
       [ True],
       [ True],
       [ True],
       [False],
       [ True],
       [ True],
       [False],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [False],
       [False],
       [False],
       [

**Explique o resultado da RNA.**

Com o limiar de 0,5, cada probabilidade é convertida em uma das duas classes. A função `evaluate` calcula a perda e a acurácia no teste.

In [14]:
rede_neural.evaluate(X_teste, y_teste, verbose=0)

[0.317484974861145, 0.8951048851013184]

## Camadas e Otimização da RNA

**Adicione mais uma camada oculta densa com 16 neurônios com a função de ativação relu e inicialize os pesos com Random uniform initializer. Qual o total de parâmetros da RNA agora?**

Com a segunda camada oculta:
- Camada 1: (30×16) + 16 = 496 parâmetros
- Camada 2: (16×16) + 16 = 272 parâmetros
- Saída: (16×1) + 1 = 17 parâmetros
- **Total: 785 parâmetros**

In [15]:
k.clear_session()
rede_neural = Sequential([
    tf.keras.layers.InputLayer(shape=(30,)),
    tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
    tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
    tf.keras.layers.Dense(units=1, activation='sigmoid')
])

In [16]:
rede_neural.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 785 (3.07 KB)

 Trainable params: 785 (3.07 KB)

 Non-trainable params: 0 (0.00 B)

**Abaixo são adicionados os parâmetros do otimizador: learning_rate e clipvalue. O que eles fazem?**

- **learning_rate (0.001)**: controla o tamanho do passo dado na descida do gradiente. Valores muito altos fazem o modelo divergir; valores muito baixos tornam o treinamento lento. 0.001 é o valor padrão do Adam.
- **clipvalue (0.5)**: limita o valor máximo dos gradientes durante o backpropagation. Evita o problema de "gradiente explosivo", onde gradientes muito grandes desestabilizam o treinamento.

In [17]:
otimizador = tf.keras.optimizers.Adam(learning_rate=0.001, clipvalue=0.5)
rede_neural.compile(optimizer=otimizador, loss='binary_crossentropy', metrics=['binary_accuracy'])

**Teste novamente a RNA. Aumentar a quantidade de camadas melhorou ou piorou os resultados? Explique o que aconteceu com a RNA e porque.**

A segunda camada aumenta a capacidade do modelo, mas não garante melhora no teste. Se o treino melhorar e o teste piorar, a rede provavelmente está sofrendo overfitting.

In [18]:
rede_neural.fit(X_treinamento, y_treinamento, batch_size=10, epochs=100, verbose=0)

In [19]:
previsoes = rede_neural.predict(X_teste, verbose=0)

In [20]:
rede_neural.evaluate(X_teste, y_teste, verbose=0)

[0.8119903802871704, 0.8741258978843689]

## K-Fold Cross Validation

**Explique como esta rede foi configurada e como é possível chegar no resultado dela. O que é necessário fazer?**

Na validação cruzada, cada fold assume uma vez o papel de validação e os demais são usados no treino. Com 10 folds, a rede é treinada 10 vezes. O `KerasClassifier` permite usar o modelo com a API do scikit-learn.

**Calcule também o Desvio Padrão dos resultados para avaliar o modelo. O que é possível concluir com esse resultado?**

O desvio padrão mostra quanto a acurácia mudou entre os folds. Valores altos indicam maior sensibilidade à divisão dos dados.

In [21]:
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import cross_val_score
from tensorflow.keras.models import Sequential
from tensorflow.keras import backend as k

In [22]:
def criar_rede():
    k.clear_session()
    rede_neural = Sequential([
        tf.keras.layers.InputLayer(shape=(30,)),
        tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
        tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
        tf.keras.layers.Dense(units=1, activation='sigmoid')
    ])
    otimizador = tf.keras.optimizers.Adam(learning_rate=0.001, clipvalue=0.5)
    rede_neural.compile(optimizer=otimizador, loss='binary_crossentropy', metrics=['binary_accuracy'])
    return rede_neural

In [23]:
rede_neural = KerasClassifier(model=criar_rede, epochs=100, batch_size=10, verbose=0)

In [24]:
resultados = cross_val_score(estimator=rede_neural, X=X, y=y, cv=10, scoring='accuracy')

In [25]:
resultados

array([0.73684211, 0.78947368, 0.85964912, 0.75438596, 0.78947368,
       0.9122807 , 0.85964912, 0.73684211, 0.84210526, 0.83928571])

In [26]:
print(f'Média de acurácia: {resultados.mean():.4f}')
print(f'Desvio padrão:     {resultados.std():.4f}')

Média de acurácia: 0.8120
Desvio padrão:     0.0565


## Overfitting e Dropout

**Como vimos, o modelo anterior sofreu overfitting. Para reduzir esse problema, aplicamos a técnica de regularização conhecida como dropout. Aplique o dropout de 20% na primeira e segunda camada oculta. O que acontece com os resultados? E o Desvio Padrão?**

O dropout desativa 20% das ativações durante o treino. A comparação deve ser feita pela média e pelo desvio padrão da validação, sem pressupor que o resultado necessariamente vai melhorar.

In [27]:
def criar_rede_dropout():
    k.clear_session()
    rede_neural = Sequential([
        tf.keras.layers.InputLayer(shape=(30,)),
        tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
        tf.keras.layers.Dropout(rate=0.2),
        tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
        tf.keras.layers.Dropout(rate=0.2),
        tf.keras.layers.Dense(units=1, activation='sigmoid')
    ])
    otimizador = tf.keras.optimizers.Adam(learning_rate=0.001, clipvalue=0.5)
    rede_neural.compile(optimizer=otimizador, loss='binary_crossentropy', metrics=['binary_accuracy'])
    return rede_neural

In [28]:
rede_neural = KerasClassifier(model=criar_rede_dropout, epochs=100, batch_size=10, verbose=0)

In [29]:
resultados = cross_val_score(estimator=rede_neural, X=X, y=y, cv=10, scoring='accuracy')

In [30]:
resultados

array([0.84210526, 0.89473684, 0.92982456, 0.77192982, 0.89473684,
       0.92982456, 0.92982456, 0.92982456, 0.8245614 , 0.96428571])

In [31]:
print(f'Média de acurácia (com Dropout): {resultados.mean():.4f}')
print(f'Desvio padrão (com Dropout):     {resultados.std():.4f}')

Média de acurácia (com Dropout): 0.8912
Desvio padrão (com Dropout):     0.0569


## Tuning dos Hiperparâmetros (técnicas de ajuste)

O ajuste de hiperparâmetros compara configurações definidas antes do treinamento.
Exemplos de hiperparâmetros:

- Taxa de aprendizado (learning rate)
- Número de árvores em um RandomForest
- Número de camadas ou neurônios em uma rede neural
- Tipo de kernel em um SVM

Algumas formas de fazer essa busca são:

- Grid Search
- Random Search
- Bayesian Optimization, etc.

**Descreva como a RNA foi configurada para fazer o processo de tuning.**

A função `criar_rede` recebe os parâmetros que serão comparados. O `KerasClassifier` adapta a rede à interface do scikit-learn, e o `GridSearchCV` avalia as combinações com cinco folds.

In [32]:
def criar_rede(optimizer, loss, kernel_initializer, activation, neurons):
    k.clear_session()
    rede_neural = Sequential([
        tf.keras.layers.InputLayer(shape=(30,)),
        tf.keras.layers.Dense(units=neurons, activation=activation, kernel_initializer=kernel_initializer),
        tf.keras.layers.Dropout(rate=0.2),
        tf.keras.layers.Dense(units=neurons, activation=activation, kernel_initializer=kernel_initializer),
        tf.keras.layers.Dropout(rate=0.2),
        tf.keras.layers.Dense(units=1, activation='sigmoid')
    ])
    rede_neural.compile(optimizer=optimizer, loss=loss, metrics=['binary_accuracy'])
    return rede_neural

In [33]:
rede_neural = KerasClassifier(model=criar_rede, verbose=0)

In [34]:
parametros = {
    'batch_size': [10, 30],
    'epochs': [50, 100],
    'model__optimizer': ['adam', 'sgd'],
    'model__loss': ['binary_crossentropy', 'hinge'],
    'model__kernel_initializer': ['random_uniform', 'normal'],
    'model__activation': ['relu', 'tanh'],
    'model__neurons': [16, 8]
}

In [35]:
# caso tiver problema com a configuração anterior, testar essa
parametros = {
    'batch_size': [10, 30],
    'epochs': [50],
    'model__optimizer': ['adam'],
    'model__loss': ['binary_crossentropy'],
    'model__kernel_initializer': ['random_uniform', 'normal'],
    'model__activation': ['relu'],
    'model__neurons': [16]
}

In [36]:
parametros

{'batch_size': [10, 30],
 'epochs': [50],
 'model__optimizer': ['adam'],
 'model__loss': ['binary_crossentropy'],
 'model__kernel_initializer': ['random_uniform', 'normal'],
 'model__activation': ['relu'],
 'model__neurons': [16]}

In [37]:
grid_search = GridSearchCV(estimator=rede_neural, param_grid=parametros,
                           scoring='accuracy', cv=5)

In [38]:
grid_search = grid_search.fit(X, y)

In [39]:
print(grid_search)

GridSearchCV(cv=5,
             estimator=KerasClassifier(model=<function criar_rede at 0x000001FE4881D9E0>, verbose=0),
             param_grid={'batch_size': [10, 30], 'epochs': [50],
                         'model__activation': ['relu'],
                         'model__kernel_initializer': ['random_uniform',
                                                       'normal'],
                         'model__loss': ['binary_crossentropy'],
                         'model__neurons': [16], 'model__optimizer': ['adam']},
             scoring='accuracy')


In [40]:
melhores_parametros = grid_search.best_params_
print(melhores_parametros)

{'batch_size': 10, 'epochs': 50, 'model__activation': 'relu', 'model__kernel_initializer': 'random_uniform', 'model__loss': 'binary_crossentropy', 'model__neurons': 16, 'model__optimizer': 'adam'}


In [41]:
melhor_precisao = grid_search.best_score_
print(melhor_precisao)

0.8981214097189877


**É possível melhorar ainda mais a acurácia da RNA? Como?**

Sim. Algumas estratégias possíveis:
- **Aumentar o espaço de busca do Grid Search**: testar mais combinações de neurônios, camadas, otimizadores e taxas de aprendizado.
- **Normalização dos dados**: aplicar StandardScaler ou MinMaxScaler nas features pode acelerar a convergência e melhorar a acurácia.
- **Aumentar épocas**: mais épocas permitem que a rede aprenda melhor, desde que combinado com dropout para evitar overfitting.
- **Adicionar mais camadas**: arquiteturas mais profundas capturam padrões mais complexos.
- **Batch Normalization**: normaliza as ativações entre camadas, estabilizando e acelerando o treinamento.
- **Early Stopping**: parar o treinamento quando a loss de validação para de melhorar, evitando overfitting.

# Para salvar o seu modelo de RNA

In [42]:
# Treina o modelo final com os melhores parâmetros antes de salvar
k.clear_session()
classificador = Sequential([
    tf.keras.layers.InputLayer(shape=(30,)),
    tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
    tf.keras.layers.Dropout(rate=0.2),
    tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
    tf.keras.layers.Dropout(rate=0.2),
    tf.keras.layers.Dense(units=1, activation='sigmoid')
])
otimizador = tf.keras.optimizers.Adam(learning_rate=0.001, clipvalue=0.5)
classificador.compile(optimizer=otimizador, loss='binary_crossentropy', metrics=['binary_accuracy'])
classificador.fit(X_treinamento, y_treinamento, batch_size=10, epochs=100, verbose=0)

In [43]:
classificador.save('classificador_breast.keras')

In [44]:
classificador_novo = tf.keras.models.load_model('classificador_breast.keras')

In [45]:
classificador_novo.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,357 (9.21 KB)

 Trainable params: 785 (3.07 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,572 (6.14 KB)